[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/klein_quadric/klein_quadric.ipynb)

# Four Lines and the Klein Quadric

This example computes the two transversal lines meeting four skew lines in three-dimensional space. Three lines in general position carry a one-parameter family of common transversals that sweeps out a hyperboloid of one sheet; a fourth line crosses that surface at two points, and through each crossing passes one line meeting all four. When the fourth line passes through the hyperboloid's waist without touching the surface, both transversals persist over the complex numbers as a conjugate pair.

A line in projective geometric algebra is modelled as a bivector meet of two planes, where a bivector represents a single line whenever `bivector ^ bivector == 0` and two lines intersect whenever `first_line ^ second_line == 0`. Both conditions come from one quadratic form on the six-dimensional bivector space, `klein = Bivector ^ Bivector`, whose null cone is called the Klein quadric. Requiring `line ^ bivector == 0` against four skew lines restricts candidates to a two-dimensional pencil of bivectors on which `klein` reduces to a scalar quadratic with two roots, while `((Point & first) ^ third) & second` builds the ruled quadric of the first three lines directly as a point-to-plane map.

In Plücker coordinates, a line is represented by six homogeneous coordinates $(p_{01}:p_{02}:p_{03}:p_{12}:p_{13}:p_{23}) \in \mathbb{P}^5$ constrained to the Klein quadric $p_{01}p_{23} - p_{02}p_{13} + p_{03}p_{12} = 0$, with incidence written as the polarized bilinear relation $p_{01}q_{23} - p_{02}q_{13} + p_{03}q_{12} + p_{23}q_{01} - p_{13}q_{02} + p_{12}q_{03} = 0$ and the count of two transversals read in Schubert calculus as the intersection product $\sigma_1^4 = 2$ on $\mathrm{Gr}(2, 4)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_states
from examples.math.klein_quadric import render
from examples.math.klein_quadric.scenarios import paced_offsets

# Complex coefficients keep both transversals present when the pair turns complex conjugate.
mv = NumpyContext(ga, np.complex128).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Every line here is a meet of two planes, an extrinsic bivector axis.
Bivector = ga.gatype.bivector()                                                # Bivector+
Point = ga.gatype.antivector()                                                 # Antivector-
# Joining a point to a line and meeting with two others maps points to intrinsic planes.
Quadric = ga.gatype((ga.gatype.antitrivector(), Point))                        # Antitrivector- <- Antivector-
# Vanishes on one bivector twice when it is a line, and on two lines when they meet.
klein = Bivector ^ Bivector                                                    # [] Pseudoscalar+ <- (Bivector+, Bivector+)
# The origin is dual to the plane at infinity, whose join of three directions meets each line at infinity.
origin = mv.w.dual()                                                           # [] Antivector-
horizon = mv.direction([1, 0, 0]) & mv.direction([0, 1, 0]) & mv.direction([0, 0, 1])   # [] Antitrivector-
# Signs selecting the two roots of a quadratic.
signs = np.array([1, -1])

## Null cone of the Klein form

The Klein form maps any pair of bivectors to their wedge product, a pseudoscalar. A meet of two planes is a single line on which the form vanishes, whereas adding two skew lines leaves a non-simple bivector with a nonzero self-wedge; similarly, two lines that intersect pair to zero while two skew lines do not.

In [ ]:
# Two coplanar lines meeting at infinity along z, and a third line skew to the first.
along_z = mv.xy                                                                # [] Bivector+
beside = (mv.x - mv.w) ^ mv.y                                                  # [] Bivector+
above = (mv.x - mv.w) ^ (mv.z - mv.w)                                          # [] Bivector+

## Transversals of four skew lines

Imposing `line ^ bivector == 0` for four skew lines leaves a two-dimensional pencil of candidate bivectors, and restricting `klein` to that pencil produces a quadratic equation whose two roots pick out the genuine lines. Three lines are placed around the waist circle of a unit hyperboloid while a steep fourth line is tested at two horizontal positions: once outside the waist where both transversals are real, and once through the waist where they form a complex-conjugate pair.

In [ ]:
# Azimuthal waist angles of the three fixed lines, rest anchor and direction of the fourth line, and two x-offsets.
waist_angles = np.array([0.3, 2.3, 4.4])
fourth_crossing, fourth_direction = [0.0, 0.4, 0.0], [0.3, -0.2, 1.0]
n_offsets = 2
offsets = np.array([1.4, 0.0])

In [ ]:
def quadratic_roots(half_linear: Scalar, constant: Scalar) -> Scalar:
    """The monic quadratic `t * t + 2 * half_linear * t + constant` has two roots (`[..., 2]
    Scalar+`), giving the weights on the first spanning element against unit weight on the second."""
    discriminant_root = (half_linear * half_linear - constant).square_root()   # [...] Scalar+
    return discriminant_root[..., None] * signs - half_linear[..., None]       # [..., 2] Scalar+


def transversals_of(lines: Bivector) -> Bivector:
    """Four lines on the trailing axis (`[..., 4] Bivector+`) determine two unit-norm transversals
    (`[..., 2] Bivector+`) meeting all four."""
    # Pairing a candidate bivector against all four lines maps the pencil of common meets to zero.
    incidence_map = (lines * klein(lines).dual()).sum(axis=-1)                 # [...] Bivector- <- Bivector+
    # The two right singular vectors of zero singular value span that pencil of bivectors.
    _, _, right_vectors = incidence_map.svd()
    pencil = right_vectors[..., -2:]                                           # [..., 2] Bivector+
    # Along the pencil the Klein form reduces to a scalar quadratic whose two zeros are genuine lines.
    first_end, second_end = pencil[..., 0], pencil[..., 1]                     # [...] Bivector+, Bivector+
    leading = klein(first_end, first_end).dual()                               # [...] Scalar-
    weights = quadratic_roots(                                                 # [..., 2] Scalar+
        klein(first_end, second_end).dual() / leading,
        klein(second_end, second_end).dual() / leading,
    )
    return (first_end[..., None] * weights + second_end[..., None]).normalized()   # [..., 2] Bivector+

In [ ]:
def fourth_lines(offsets: Scalar) -> Bivector:
    """Translating the anchor along x by `offsets` yields the fourth line (`[...] Bivector+`) as the
    meet of two coordinate-aligned planes through it."""
    shifts = (mv.xw * (offsets / 2)).exp()                                     # [...] Motor+
    anchors = shifts >> mv.point(fourth_crossing)                              # [...] Antivector-
    # Joining anchor and direction gives the intrinsic line, then meeting two direction-spanned planes gives its extrinsic axis.
    intrinsic_line = anchors & mv.direction(fourth_direction)                  # [...] Antibivector-
    return (intrinsic_line & mv.direction([1, 0, 0])) ^ (intrinsic_line & mv.direction([0, 1, 0]))   # [...] Bivector+


# Turning one ruling of the unit hyperboloid about z places three skew lines around its waist.
leaning = (mv.x - mv.w) ^ (mv.y - mv.z)                                        # [] Bivector+
waist_turns = (mv.xy * (-waist_angles / 2)).exp()                              # [3] Rotor+
three_lines = waist_turns >> leaning                                           # [3] Bivector+
sample_fourth = fourth_lines(offsets)                                          # [n_offsets] Bivector+
sample_lines = stack(                                                          # [n_offsets, 4] Bivector+
    [three_lines[0].broadcast_to(sample_fourth.shape), three_lines[1].broadcast_to(sample_fourth.shape),
     three_lines[2].broadcast_to(sample_fourth.shape), sample_fourth],
    axis=-1,
)
sample_transversals = transversals_of(sample_lines)                            # [n_offsets, 2] Bivector+

## Ruled quadric of three lines

The lines meeting three skew lines sweep out a ruled quadric whose point-to-plane map `((Point & first) ^ third) & second` pairs to zero with every point on the surface: joining a point to the first line gives a plane meeting the third line at a point, and joining that intersection to the second line produces a plane that contains the starting point whenever a single transversal meets all three. Intersecting the fourth line with that quadric yields a quadratic along the fourth line whose two crossing points lie on the two transversals.

In [ ]:
# Ruling count per family, animation frame count, swing amplitude, pacing resolution, still index, and frame delay.
n_rulings = 12
n_frames = 168
swing_amplitude = 1.5
n_path, line_share = 960, 0.6
still_frame = n_frames // 12
frame_duration_ms = 40

In [ ]:
def crossings_of(surface: Quadric, line: Bivector) -> Point:
    """A line (`[...] Bivector+`) crosses `surface` at two unit-weight points (`[..., 2]
    Antivector-`)."""
    # Every finite line is spanned by its point nearest the origin and its ideal direction point.
    nearest = (origin | line) ^ line                                           # [...] Antivector-
    direction = line ^ horizon                                                 # [...] Antivector-
    # Averaging both orders extracts the symmetric cross term even when `surface` is unsymmetrized.
    cross = ((surface(nearest) & direction) + (surface(direction) & nearest)) / 2   # [...] Scalar-
    leading = surface(nearest) & nearest                                       # [...] Scalar-
    weights = quadratic_roots(cross / leading, (surface(direction) & direction) / leading)   # [..., 2] Scalar+
    crossings = nearest[..., None] * weights + direction[..., None]            # [..., 2] Antivector-
    return crossings.normalized()                                              # [..., 2] Antivector-


def line_points(line: Bivector, n_points: int) -> Point:
    """Homogeneous combinations of the unit-weight point nearest the origin and the unit ideal
    direction sample `n_points` points (`[..., n_points] Antivector-`) around the full projective
    line."""
    unit_line = line.normalized()                                              # [...] Bivector+
    nearest = ((origin | unit_line) ^ unit_line).normalized()                  # [...] Antivector-
    direction = unit_line ^ horizon                                            # [...] Antivector-
    angles = np.linspace(0.0, np.pi, n_points, endpoint=False)
    return nearest[..., None] * np.cos(angles) + direction[..., None] * np.sin(angles)   # [..., n_points] Antivector-


surface = ((Point & three_lines[0]) ^ three_lines[2]) & three_lines[1]         # [] Antitrivector- <- Antivector-
# Meeting the two planes joining points along one line to the other two builds the first ruling family, and repeating builds the second.
starts = line_points(three_lines[0], n_rulings)                                # [n_rulings] Antivector-
transversal_family = (starts & three_lines[1]) ^ (starts & three_lines[2])     # [n_rulings] Bivector+
second_starts = line_points(transversal_family[0], n_rulings)                  # [n_rulings] Antivector-
original_family = (                                                            # [n_rulings] Bivector+
    (second_starts & transversal_family[n_rulings // 3])
    ^ (second_starts & transversal_family[2 * n_rulings // 3])
)
rulings = stack([transversal_family, original_family])                         # [2, n_rulings] Bivector+

In [ ]:
# Signed crossing separation along a dense cosine path paces the swing so tangency is resolved cleanly.
path = swing_amplitude * np.cos(np.linspace(0.0, 2 * np.pi, n_path + 1))
path_crossings = crossings_of(surface, fourth_lines(path))                     # [n_path + 1, 2] Antivector-
separations = (path_crossings[..., 0] & path_crossings[..., 1]).scalar_norm_squared()   # [n_path + 1] Scalar+
frame_offsets = paced_offsets(path, separations, n_frames, line_share)         # [n_frames] Scalar+


def swing_scenes(frame_offsets: Scalar) -> Iterator[tuple[Bivector, Bivector, Bivector, Point]]:
    """Yields `(lines, rulings, transversals, crossings)` tuples across `frame_offsets`."""
    for index in range(len(frame_offsets)):
        fourth_line = fourth_lines(frame_offsets[index])                       # [] Bivector+
        frame_lines = stack([three_lines[0], three_lines[1], three_lines[2], fourth_line])   # [4] Bivector+
        yield frame_lines, rulings, transversals_of(frame_lines), crossings_of(surface, fourth_line)


recorded_scenes = show_states(
    swing_scenes(frame_offsets),
    render.swing_frames,
    n_frames,
    name="klein_swing",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
)

In [ ]:
still_figure = render.scene_figure(*recorded_scenes[still_frame])
save_figure(still_figure, "klein_transversals");

The three fixed skew lines in dark blue generate the faint blue and grey ruling families of the hyperboloid, while the fourth line in orange swings into the waist and back out. Outside the waist the orange line pierces the surface at two red crossing balls, each carrying a red transversal that meets all four lines; pacing slows the swing near tangency so the two crossings and their red transversals are seen to slide together, merge on the surface, and vanish into a complex-conjugate pair across the waist before re-emerging on the return stroke.

In [ ]:
# checks
# The Klein form vanishes on a line and on intersecting lines, and stays nonzero on non-simple bivectors and skew pairs.
assert klein(along_z, along_z).dual().abs() < 1e-12
assert klein(along_z, beside).dual().abs() < 1e-12
assert klein(along_z + above, along_z + above).dual().abs() > 0.5
assert klein(along_z, above).dual().abs() > 0.5
# Transversals are genuine lines meeting all four; outside the waist both are real, through the waist both are complex.
assert ((sample_transversals ^ sample_transversals).dual().abs() < 1e-11).all()
assert (((sample_lines[..., :, None] ^ sample_transversals[..., None, :]).dual()).abs() < 1e-11).all()
is_real = np.abs(sample_transversals.kernel.imag).max(axis=-1) <= 1e-6 * np.abs(sample_transversals.kernel).max(axis=-1)
assert is_real[0].all() and not is_real[1].any()
# Streamed swing crossings lie on the quadric and on the transversals, and both ruling families lie on the quadric.
swing_transversals = stack([state[2] for state in recorded_scenes])            # [n_frames, 2] Bivector+
swing_crossings = stack([state[3] for state in recorded_scenes])               # [n_frames, 2] Antivector-
assert ((surface(swing_crossings) & swing_crossings).abs() < 1e-9).all()
incidence_residuals = np.abs((swing_crossings[:, :, None] & swing_transversals[:, None, :]).kernel).max(axis=-1)
assert (incidence_residuals.min(axis=-1) < 1e-9).all()
ruling_samples = line_points(rulings, 3)                                       # [2, n_rulings, 3] Antivector-
assert ((surface(ruling_samples) & ruling_samples).abs() < 1e-9).all()